In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/processed/clean.csv")
df = df.dropna(subset=["clean"])
print(df.shape)

train, test = train_test_split(df, test_size=0.2,
                               stratify=df["label"], random_state=42)
y_train, y_test = train["label"], test["label"]

print(len(train), len(test))
print(y_train.mean(), y_test.mean())   # should both be about 0.87

(79788, 7)
63830 15958
0.8676954410151966 0.867715252537912


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), min_df=5)
X_train = tfidf.fit_transform(train["clean"])
X_test = tfidf.transform(test["clean"])
print(X_train.shape, X_test.shape)

(63830, 50000) (15958, 50000)


In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score

models = {
    "Always positive": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Naive Bayes": MultinomialNB(),
}

results = []
for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    results.append({"model": name,
                    "accuracy": round(accuracy_score(y_test, pred), 4),
                    "macro_f1": round(f1_score(y_test, pred, average="macro"), 4)})

pd.DataFrame(results)

,model,accuracy,macro_f1
0,Always positive,0.8677,0.4646
1,Logistic regression,0.9004,0.8126
2,Naive Bayes,0.8781,0.5397


In [4]:
from sklearn.metrics import classification_report, confusion_matrix

lr = models["Logistic regression"]
pred = lr.predict(X_test)
print(classification_report(y_test, pred, target_names=["negative", "positive"]))
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

    negative       0.59      0.82      0.68      2111
    positive       0.97      0.91      0.94     13847

    accuracy                           0.90     15958
   macro avg       0.78      0.86      0.81     15958
weighted avg       0.92      0.90      0.91     15958

[[ 1722   389]
 [ 1200 12647]]


In [5]:
from sklearn.naive_bayes import ComplementNB

cnb = ComplementNB()
cnb.fit(X_train, y_train)
pred_cnb = cnb.predict(X_test)
print("ComplementNB  acc:", round(accuracy_score(y_test, pred_cnb), 4),
      " macro-F1:", round(f1_score(y_test, pred_cnb, average="macro"), 4))

ComplementNB  acc: 0.9164  macro-F1: 0.7963


In [6]:
test = test.copy()
test["pred"] = pred          # logistic regression predictions from cell 4

fp = test[(test["label"] == 1) & (test["pred"] == 0)]   # actually positive, flagged negative
print("Positive reviews flagged negative:", len(fp))
print(fp["score"].value_counts().sort_index())           # how many were 4 vs 5 stars?

for t in fp["text"].sample(5, random_state=2):
    print("-", t[:300], "\n")

Positive reviews flagged negative: 1200
score
4.0    489
5.0    711
Name: count, dtype: int64
- Wow. If I could write, that would be a book for which I wouldstrive. Transending age, utilizing Arch-types and new ideas, anyonecan enjoy this book.The idea of a no-one becoming a Someone is a great pleasure to all of us no-ones out there. Harry is smart, but not brainy (that is H.'s job). He is goo 

- Being that this is a review, and not a long-winded critique, I would first address the central premise of this novel, stated by John Galt, which essentially praises the virtues of selfishness, and the evils of what Rand considers altruism. The Websters Dictionary defines altruisim as unselfish conce 

- You cannot blame Dostoyevsky for NOT being a &quot;beach read.&quot; If you read this book in two weeks (and people praise themselves for it! how ironic) and got nothing out of it... that's your problem! This is not TOm CLancy and all that garbage. This is a book you must meditate on for a whi

In [7]:
import numpy as np

names = np.array(tfidf.get_feature_names_out())
coefs = lr.coef_[0]                 # positive coefficient = pushes toward "positive"
order = np.argsort(coefs)

print("MOST NEGATIVE:", list(names[order[:30]]))
print()
print("MOST POSITIVE:", list(names[order[-30:]][::-1]))

MOST NEGATIVE: ['boring', 'not', 'disappointed', 'bad', 'waste', 'not recommend', 'disappointment', 'unfortunately', 'instead', 'disappointing', 'poorly', 'not worth', 'no', 'poor', 'lack', 'dull', 'suppose', 'bore', 'claim', 'fail', 'page', 'horrible', 'not good', 'useless', 'author', 'sorry', 'not buy', 'waste time', 'maybe', 'terrible']

MOST POSITIVE: ['great', 'excellent', 'wonderful', 'amazing', 'love', 'good', 'not wait', 'easy', 'favorite', 'not disappoint', 'highly recommend', 'life', 'fantastic', 'highly', 'enjoy', 'read', 'recommend', 'fun', 'powerful', 'awesome', 'beautiful', 'fascinating', 'perfect', 'definitely', 'thank', 'superb', 'not want', 'help', 'love book', 'brilliant']


In [8]:
from gensim.models import Word2Vec
import numpy as np

sentences = [t.split() for t in train["clean"]]
w2v = Word2Vec(sentences, vector_size=100, window=5, min_count=5,
               workers=4, epochs=10, seed=42)

print("vocabulary size:", len(w2v.wv))
print(w2v.wv.most_similar("ending", topn=5))
print(w2v.wv.most_similar("boring", topn=5))
print(w2v.wv.most_similar("character", topn=5))

vocabulary size: 28113
[('end', 0.7448205947875977), ('climax', 0.6597234606742859), ('abrupt', 0.6173906922340393), ('cliffhanger', 0.6155519485473633), ('resolution', 0.6064832210540771)]
[('dull', 0.8022202253341675), ('bore', 0.7990142107009888), ('tedious', 0.7478922605514526), ('pointless', 0.7146562933921814), ('bored', 0.6841341257095337)]
[('charater', 0.7755118012428284), ('charecter', 0.7377369999885559), ('charcter', 0.7200002074241638), ('charachter', 0.7198783159255981), ('charactor', 0.6461700797080994)]


In [9]:
def doc_vec(text):
    vecs = [w2v.wv[w] for w in text.split() if w in w2v.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(100)

Xw_train = np.vstack(train["clean"].map(doc_vec).tolist())
Xw_test = np.vstack(test["clean"].map(doc_vec).tolist())

lr_w2v = LogisticRegression(max_iter=1000, class_weight="balanced")
lr_w2v.fit(Xw_train, y_train)
pred_w2v = lr_w2v.predict(Xw_test)
print("Word2Vec + LogReg  acc:", round(accuracy_score(y_test, pred_w2v), 4),
      " macro-F1:", round(f1_score(y_test, pred_w2v, average="macro"), 4))

Word2Vec + LogReg  acc: 0.8213  macro-F1: 0.7195
